# Pré Execução

In [ ]:
%pip install -r requirements.txt

# Execução

In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords
from pathlib import Path
import spacy

In [2]:
FILENAME = "data\\amazon.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(FILENAME, delimiter=",")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   product_id           1465 non-null   str  
 1   product_name         1465 non-null   str  
 2   category             1465 non-null   str  
 3   discounted_price     1465 non-null   str  
 4   actual_price         1465 non-null   str  
 5   discount_percentage  1465 non-null   str  
 6   rating               1465 non-null   str  
 7   rating_count         1463 non-null   str  
 8   about_product        1465 non-null   str  
 9   user_id              1465 non-null   str  
 10  user_name            1465 non-null   str  
 11  review_id            1465 non-null   str  
 12  review_title         1465 non-null   str  
 13  review_content       1465 non-null   str  
 14  img_link             1465 non-null   str  
 15  product_link         1465 non-null   str  
dtypes: str(16)
memory usage: 183.3 KB


In [3]:
df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


## Data Cleaning

<ul>
<li>Padronização da Moeda</li>
<li>Padronização da porcentagem</li>
<li>Padronização do Rating</li>
<li>Padronização da contagem de ratings</li>
<li>Remover duplicados</li>
<li>Segmentar Categorias</li>
</ul>

Remoção de duplicadas

### Padronização do texto das reviews

In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text) # Remove URLs
    text = re.sub(r"@\w+", "", text) # Remove mentions
    text = re.sub(r"[^\w\s,]", " ", text) # Remove punctuation except commas
    text = re.sub(r"\d+", "", text) # Remove numbers
    text = re.sub(r"\s+", " ", text).strip() # Remove extra spaces
    return text  

In [5]:
df["review_content"] = df["review_content"].fillna("")
df["review_title"] = df["review_title"].fillna("")

In [6]:
apply_stopwords_cols = ["review_content", "review_title", "about_product", "product_name"]
stop_words = set(stopwords.words("english"))

for col in apply_stopwords_cols:
    df[col] = df[col].apply(
        lambda x: " ".join(
            word for word in str(x).split()
            if word not in stop_words
        )
    )

In [7]:
nlp = spacy.load("en_core_web_sm")
def lemmatize(text):
    doc = nlp(text)
    return " ".join(token.lemma_ for token in doc)

In [8]:
for i in range(len(df)):
    df.loc[i, "review_content"] = lemmatize(clean_text(df.iloc[i]["review_content"]))
    df.loc[i, "review_title"] = lemmatize(clean_text(df.iloc[i]["review_title"]))
    df.loc[i, "about_product"] = lemmatize(clean_text(df.iloc[i]["about_product"]))
    df.loc[i, "product_name"] = lemmatize(clean_text(df.iloc[i]["product_name"]))

In [9]:
temp_ls = df["review_content"].to_string(header=False, index=False).split()
temp_output = []

for word in temp_ls:
    if word.lower() not in temp_output:
        temp_output.append(word.lower())

temp_output.sort()
print(temp_output)

[',', ',...', '...', '_', 'a', 'a...', 'aa...', 'aacha', 'ab...', 'ability', 'able', 'about', 'abroad', 'absolutely', 'ac', 'ac...', 'accessory', 'accord', 'accu...', 'accuracy', 'accurat...', 'accurate', 'accurately', 'acha', 'achcha', 'achha', 'aco...', 'activi...', 'actually', 'add', 'addition...', 'adequate', 'adhesion', 'adhesive', 'adhesive...', 'adjustment', 'adjustment...', 'adpeator', 'adult', 'advanced', 'advertise', 'advise', 'aesthetically', 'affair', 'affix', 'affor...', 'affordable', 'after', 'aftr', 'age', 'ago', 'air', 'airtel', 'airtel...', 'al...', 'alexa', 'alignment', 'all', 'all...', 'almost', 'along', 'alot', 'alptop', 'als...', 'also', 'alternative', 'although', 'always', 'ama...', 'amaz...', 'amazing', 'amazingly', 'amazon', 'amazonbasic', 'amazone', 'amazonthank', 'ambrane', 'amd', 'ame...', 'amount', 'amp', 'amul', 'an', 'and', 'and...', 'android', 'another', 'anyone...', 'anyt...', 'apart', 'app', 'app...', 'apparatus', 'appeal', 'appear', 'appl...', 'apple',

Padronização de rating

In [10]:
# Padronização de rating
df["rating"] = df["rating"].replace("|", "0")
df["rating"] = df["rating"].astype(float)

Padronização de moeda

In [11]:
# Padronização de moeda
df["actual_price"] = (df["actual_price"]
                      .str.replace("₹", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .astype(float))

df["discounted_price"] = (df["discounted_price"]
                      .str.replace("₹", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .astype(float))

Padronização de Porcentagens

In [12]:
# Padronização da porcentagem
df["discount_percentage"] = (df["discount_percentage"]
                      .str.replace("%", "", regex=False)
                      .str.replace(".", "", regex=False)
                      .astype(float))

df["discount_percentage"] = df["discount_percentage"]/100


Padronização do rating count

In [13]:
# Padronização do rating count
df["rating_count"] = df["rating_count"].fillna("0,0")
df["rating_count"] = (df["rating_count"]
                      .str.replace(",", "", regex=False)
                      .astype(float))

In [14]:
# Remoção de colunas não analizáveis
df=df.drop(["img_link", "product_link"], axis=1)

In [15]:
df = df.fillna("")

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           1465 non-null   str    
 1   product_name         1465 non-null   str    
 2   category             1465 non-null   str    
 3   discounted_price     1465 non-null   float64
 4   actual_price         1465 non-null   float64
 5   discount_percentage  1465 non-null   float64
 6   rating               1465 non-null   float64
 7   rating_count         1465 non-null   float64
 8   about_product        1465 non-null   str    
 9   user_id              1465 non-null   str    
 10  user_name            1465 non-null   str    
 11  review_id            1465 non-null   str    
 12  review_title         1465 non-null   str    
 13  review_content       1465 non-null   str    
dtypes: float64(5), str(9)
memory usage: 160.4 KB


### Divisão em tabelas

#### Produtos

In [17]:
produtos_df = df[["product_id", "product_name", "category", "about_product", "actual_price", "discounted_price", "discount_percentage", "rating", "rating_count"]]

In [18]:
# Segmentação de categorias
for index, row in produtos_df.iterrows():
    if isinstance(row["category"], str):
        temp = row["category"].split("|")
        for i in range(len(temp)):
            produtos_df.loc[index, f"category_{i+1}"] = temp[i]

produtos_df = produtos_df.drop("category", axis=1)

In [19]:
produtos_df = produtos_df.drop_duplicates("product_id")
produtos_df = produtos_df.fillna("")

In [20]:
produtos_df.info()

<class 'pandas.DataFrame'>
Index: 1351 entries, 0 to 1464
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           1351 non-null   str    
 1   product_name         1351 non-null   str    
 2   about_product        1351 non-null   str    
 3   actual_price         1351 non-null   float64
 4   discounted_price     1351 non-null   float64
 5   discount_percentage  1351 non-null   float64
 6   rating               1351 non-null   float64
 7   rating_count         1351 non-null   float64
 8   category_1           1351 non-null   str    
 9   category_2           1351 non-null   str    
 10  category_3           1351 non-null   str    
 11  category_4           1351 non-null   str    
 12  category_5           1351 non-null   str    
 13  category_6           1351 non-null   str    
 14  category_7           1351 non-null   str    
dtypes: float64(5), str(10)
memory usage: 168.9 KB


#### Clientes

In [21]:
clientes_df = df[["user_id","user_name"]]

In [22]:
clients = {}
for index, row in clientes_df.iterrows():
    temp_id = row["user_id"].split(",")
    temp_name = row["user_name"].split(",")
    for i, j in zip(temp_id, temp_name):
        clients[i] = j

In [23]:
clientes_df = pd.DataFrame(list(clients.items()), columns=["user_id", "user_name"])

In [24]:
clientes_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9050 entries, 0 to 9049
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   user_id    9050 non-null   str  
 1   user_name  9050 non-null   str  
dtypes: str(2)
memory usage: 141.5 KB


#### Reviews

In [25]:
reviews_df = df[["review_id", "user_id", "product_id", "review_title", "review_content"]]

In [26]:
reviews = {}
for index, row in reviews_df.iterrows():
    temp_id = row["review_id"].split(",")
    temp_user_id = row["user_id"].split(",")
    temp_review_title = row["review_title"].split(",")
    temp_review_content = row["review_content"].split(",")

    max_len = min(len(temp_id), len(temp_user_id), len(temp_review_title), len(temp_review_content))

    for i in range(max_len):
        review_id = temp_id[i]
        if review_id not in reviews:
            reviews[review_id] = {
                "user_id": temp_user_id[i],
                "review_title": temp_review_title[i],
                "review_content": temp_review_content[i],
            }

reviews_df = pd.DataFrame(
    [
        {
            "review_id": review_id,
            "user_id": values["user_id"],
            "review_title": values["review_title"],
            "review_content": values["review_content"],
        }
        for review_id, values in reviews.items()
    ]
)

In [27]:
reviews_df = reviews_df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
reviews_df = reviews_df.fillna("")

In [28]:
reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9222 entries, 0 to 9221
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   review_id       9222 non-null   str  
 1   user_id         9222 non-null   str  
 2   review_title    9222 non-null   str  
 3   review_content  9222 non-null   str  
dtypes: str(4)
memory usage: 288.3 KB


In [29]:
reviews_df.head()

,review_id,user_id,review_title,review_content
0,R3HXWT0LRP0NMF,AG3D6O4STAQKAY2UVGEUV46KN35Q,satisfied,look durable charge fine toono complain
1,R2AJM3LFTLZHFO,AHMY5CWJMMK5BJRBBSNLYT3ONILA,charge really fast,charge really fast
2,R6AQJGUP6P86,AHCTC6ULH4XB6YHDY6PCH2R772LQ,value money,good product
3,R1KD19VHEDV0OR,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,product review,till satisfied quality
4,R3C02RMYQMK6FC,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,good quality,this good product the charge speed slow origi...


### Sentiment Analyzis

In [30]:
positive_words = [
    "good", "great", "excellent", "love", "satisfied", "nice", "amazing",
    "worth", "fast", "durable", "sturdy", "quality", "value", "reliable",
    "works", "well", "perfect", "recommend", "premium", "strong"
]
negative_words = [
    "bad", "poor", "slow", "broken", "issue", "problem", "disconnect",
    "not working", "useless", "waste", "damage", "loose", "faulty",
    "lag", "bad quality", "stop work", "not support", "frustrating"
]

reason_keywords = {
    "charging": ["charge", "charging", "battery", "power", "adapter", "slow", "fast charge"],
    "durability": ["durable", "sturdy", "broke", "damage", "stop work", "loose", "break", "last long"],
    "price": ["price", "value", "cheap", "expensive", "worth", "budget", "cost"],
    "compatibility": ["compatible", "connect", "carplay", "android auto", "device", "port", "fit"],
    "service": ["delivery", "service", "seller", "amazon", "replacement", "install", "support"],
    "quality": ["quality", "build", "material", "finish", "sound", "picture", "screen"],
    "performance": ["speed", "lag", "performance", "signal", "data transfer", "transfer"],
}


def classify_sentiment(text):
    text = str(text).lower()
    pos_score = sum(text.count(word) for word in positive_words)
    neg_score = sum(text.count(word) for word in negative_words)

    if pos_score == 0 and neg_score == 0:
        return "neutral", 0.0

    score = (pos_score - neg_score) / (pos_score + neg_score)
    if score > 0.15:
        return "positive", round(score, 3)
    elif score < -0.15:
        return "negative", round(score, 3)
    return "neutral", round(score, 3)

In [31]:
def infer_reason(text):
    text = str(text).lower()
    matches = []

    for category, keywords in reason_keywords.items():
        if any(keyword in text for keyword in keywords):
            matches.append(category)

    if not matches:
        return "general", "general feedback"

    primary_reason = matches[0]
    return primary_reason, f"review mentions {primary_reason} related feedback"


In [32]:
reviews_df["review_text"] = (
    reviews_df["review_title"].fillna("") + " " + reviews_df["review_content"].fillna("")
).str.lower()

reviews_df[["sentiment_label", "sentiment_score"]] = reviews_df["review_text"].apply(
    lambda x: pd.Series(classify_sentiment(x))
)

reviews_df[["reason_category", "reason_detail"]] = reviews_df["review_text"].apply(
    lambda x: pd.Series(infer_reason(x))
)

reviews_df = reviews_df.drop(columns=["review_text"])
reviews_df.head()

,review_id,user_id,review_title,review_content,sentiment_label,sentiment_score,reason_category,reason_detail
0,R3HXWT0LRP0NMF,AG3D6O4STAQKAY2UVGEUV46KN35Q,satisfied,look durable charge fine toono complain,positive,1.0,charging,review mentions charging related feedback
1,R2AJM3LFTLZHFO,AHMY5CWJMMK5BJRBBSNLYT3ONILA,charge really fast,charge really fast,positive,1.0,charging,review mentions charging related feedback
2,R6AQJGUP6P86,AHCTC6ULH4XB6YHDY6PCH2R772LQ,value money,good product,positive,1.0,price,review mentions price related feedback
3,R1KD19VHEDV0OR,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,product review,till satisfied quality,positive,1.0,quality,review mentions quality related feedback
4,R3C02RMYQMK6FC,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,good quality,this good product the charge speed slow origi...,positive,0.5,charging,review mentions charging related feedback


## Export em csv para PowerBI

In [33]:
df.to_csv("outputs\\amazon_cleaned.csv", sep=",", index=False)
produtos_df.to_csv("outputs\\produtos_tb.csv", sep=",", index=False)
clientes_df.to_csv("outputs\\clientes_tb.csv", sep=",", index=False)
reviews_df.to_csv("outputs\\reviews_tb.csv", sep=",", index=False)